<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
import numpy as np
import torch

# Example of Torch

### How to use (p. 6)


In [79]:
x = torch.tensor([[10., 2., 0.], [ 0., -1., 1.]])
w = torch.tensor([[1.], [0.], [1.]], requires_grad=True)
y = torch.tensor([[ 1.], [-1.]])
logit = x @ w # 행렬 곱 (logit = x * w)
margin = logit * y # 원소별 곱 (margin = logit ⊙ y)
prob = torch.sigmoid(margin) # 로지스틱/시그모이드 함수 (logistic)
log_prob = torch.log(prob) # 로그 (log) #
total_loss = -torch.mean(log_prob) # -ones/N * log_prob
print(f"total_loss = {total_loss.item():.3f}")

total_loss = 0.657


In [80]:
total_loss.backward()  # 연산 그래프를 따라 w의 기울기 자동 계산

print(f"Total Loss: {total_loss.item():.3f}\n")
print("w에 대한 기울기 (w.grad):")
print(w.grad)

Total Loss: 0.657

w에 대한 기울기 (w.grad):
tensor([[-2.2709e-04],
        [-3.6557e-01],
        [ 3.6553e-01]])


### Page 9

In [81]:
x = torch.tensor(1., requires_grad=True)
y = x ** 2
z = y ** 2
u = torch.tensor(3., requires_grad=True)
l2 = y.detach() ** 2 + u
l2.backward()

print(f"l2 = {l2.item():.3f}")
print(f"u.grad = {u.grad:.3f}")

l2 = 4.000
u.grad = 1.000


In [82]:
### print(f"x.grad = {x.grad:.3f}") ### ERROR!!!

In [83]:
z.backward()
print(f"x.grad = {x.grad:.3f}")

x.grad = 4.000


### Page 11

In [84]:
with torch.no_grad():
  y = x ** 2
  z = y ** 2

In [85]:
try:
  z.backward()
except RuntimeError as e:
  print(e)

element 0 of tensors does not require grad and does not have a grad_fn


### Page 12

In [86]:
x = torch.tensor([1., 2, 3, 4])
target_y = torch.tensor([0., 1, 0])
model = nn.Linear(4, 3)
logits = model(x)
cross_entropy = nn.CrossEntropyLoss()
loss = cross_entropy(logits, target_y)
loss.backward()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) # SGD = stochastic gradient descent
optimizer.step() # Updates the parameters


### Page 13

In [87]:
import torch
import torch.nn as nn
import altair as alt

class Example:
  def __init__(self, x: torch.Tensor, target_y: torch.Tensor):
    self.x = x
    self.target_y = target_y

def get_training_data() -> list[Example]:
  return [
    Example(x=torch.tensor([1., 2, 0, 1]), target_y=torch.tensor([0., 1, 0])),
    Example(x=torch.tensor([-1., 0, 2, 0]), target_y=torch.tensor([1., 0, 0])),
    Example(x=torch.tensor([0., 3, 1, 0]), target_y=torch.tensor([0., 0, 1])),
  ]

def train_model(model: nn.Module,
                training_data: list[Example],
                optimizer_class=torch.optim.SGD,
                num_steps=500,
                learning_rate=0.1):
  """Train the model on `training_data`."""
  # Create data in tensor format (every row is an example)
  x = torch.stack([example.x for example in training_data])
  target_y = torch.stack([example.target_y for example in training_data])
  cross_entropy = nn.CrossEntropyLoss()
  losses: list[float] = []
  optimizer = optimizer_class(model.parameters(), lr=learning_rate)
  for step in range(num_steps):
    # Forward pass
    logits = model(x)
    loss = cross_entropy(logits, target_y)
    losses.append(loss.item())
    # Backward pass
    optimizer.zero_grad()  # Remember to do this!
    loss.backward()
    # Update parameters
    optimizer.step()
    parameters = list(model.named_parameters())
  return alt.Chart(alt.Data(values=[{"step": i, "loss": loss} for i, loss in enumerate(losses)])).mark_line().encode(x="step:Q", y="loss:Q").to_dict()

In [88]:
training_data = get_training_data()
result = train_model(model, training_data)

In [89]:
chart = alt.Chart.from_dict(result)
chart.show()

alt.Chart(...)

In [90]:
result

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}},
 'data': {'values': [{'step': 0, 'loss': 1.2850056886672974},
   {'step': 1, 'loss': 1.1156963109970093},
   {'step': 2, 'loss': 0.9816503524780273},
   {'step': 3, 'loss': 0.8768126368522644},
   {'step': 4, 'loss': 0.794288694858551},
   {'step': 5, 'loss': 0.7280750870704651},
   {'step': 6, 'loss': 0.6736821532249451},
   {'step': 7, 'loss': 0.6279916763305664},
   {'step': 8, 'loss': 0.5888820290565491},
   {'step': 9, 'loss': 0.5548940300941467},
   {'step': 10, 'loss': 0.5249958038330078},
   {'step': 11, 'loss': 0.49843382835388184},
   {'step': 12, 'loss': 0.47464117407798767},
   {'step': 13, 'loss': 0.453179270029068},
   {'step': 14, 'loss': 0.4337019920349121},
   {'step': 15, 'loss': 0.4159306287765503},
   {'step': 16, 'loss': 0.39963826537132263},
   {'step': 17, 'loss': 0.3846374452114105},
   {'step': 18, 'loss': 0.3707720935344696},
   {'step': 19, 'loss': 0.3579108417034149},
   {'step': 20, 'lo

### Page 21

In [91]:
import torch.nn.functional as F

class MultiLayerPerceptron(nn.Module):
  def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
    super().__init__()
    # Maps input to hidden layer pre-nonlinearity
    self.w1 = nn.Linear(input_dim, hidden_dim)
    # Maps hidden layer to output logits
    self.w2 = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    # Maps input to hidden layer (learned feature map)
    x_transformed = self.w1(x)
    hidden = F.relu(x_transformed)
    # Maps hidden layer to output logits
    logits = self.w2(hidden)
    return logits
  def asdict(self):
    return list(self.named_parameters())

In [92]:
training_data = get_training_data()
input_dim = len(training_data[0].x)
num_classes = len(training_data[0].target_y)
torch.manual_seed(1)
model = MultiLayerPerceptron(input_dim=input_dim, hidden_dim=5, num_classes=num_classes)
logits = model(training_data[0].x)
result = train_model(model, training_data)

In [93]:
chart = alt.Chart.from_dict(result)
chart.show()

alt.Chart(...)

In [94]:
result

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}},
 'data': {'values': [{'step': 0, 'loss': 1.0769004821777344},
   {'step': 1, 'loss': 1.0516403913497925},
   {'step': 2, 'loss': 1.0384384393692017},
   {'step': 3, 'loss': 1.0246431827545166},
   {'step': 4, 'loss': 1.0101795196533203},
   {'step': 5, 'loss': 0.9949910044670105},
   {'step': 6, 'loss': 0.9791514277458191},
   {'step': 7, 'loss': 0.9630721211433411},
   {'step': 8, 'loss': 0.9466055035591125},
   {'step': 9, 'loss': 0.9290353655815125},
   {'step': 10, 'loss': 0.9114242196083069},
   {'step': 11, 'loss': 0.8925895690917969},
   {'step': 12, 'loss': 0.8741037249565125},
   {'step': 13, 'loss': 0.8543890118598938},
   {'step': 14, 'loss': 0.8354454040527344},
   {'step': 15, 'loss': 0.8161132335662842},
   {'step': 16, 'loss': 0.7970380783081055},
   {'step': 17, 'loss': 0.7775099873542786},
   {'step': 18, 'loss': 0.7589506506919861},
   {'step': 19, 'loss': 0.7404257655143738},
   {'step': 20, 'los

### Page 26

In [97]:
x = torch.tensor(1.)
w = torch.tensor(0.5, requires_grad=True)
for layer in range(20):
  x = x * w
x.backward()
w.grad

tensor(3.8147e-05)

In [98]:
x = torch.tensor(1.)
w = torch.tensor(2.0, requires_grad=True)
for layer in range(20):
  x = x * w
x.backward()
w.grad

tensor(10485760.)

### Page 27

In [102]:
class DNNWithResidual(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.w1 = nn.Linear(input_dim, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, hidden_dim)
        self.w3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.w1(x))
        x = x + F.relu(self.w2(x))
        x = self.w3(x)
        return x
    def asdict(self):
        return list(self.named_parameters())

In [103]:
# Data
training_data = get_training_data()
input_dim = len(training_data[0].x)
num_classes = len(training_data[0].target_y)
# Model
model = DNNWithResidual(input_dim=input_dim, hidden_dim=5, num_classes=num_classes)
logits = model(training_data[0].x)
# Train
result = train_model(model, training_data)

In [104]:
chart = alt.Chart.from_dict(result)
chart.show()

alt.Chart(...)

In [105]:
result

{'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}},
 'data': {'values': [{'step': 0, 'loss': 1.271896481513977},
   {'step': 1, 'loss': 1.155785083770752},
   {'step': 2, 'loss': 1.0731147527694702},
   {'step': 3, 'loss': 1.0106241703033447},
   {'step': 4, 'loss': 0.9591808319091797},
   {'step': 5, 'loss': 0.9179635047912598},
   {'step': 6, 'loss': 0.8765249252319336},
   {'step': 7, 'loss': 0.8397791981697083},
   {'step': 8, 'loss': 0.7966125011444092},
   {'step': 9, 'loss': 0.7613363265991211},
   {'step': 10, 'loss': 0.7197586894035339},
   {'step': 11, 'loss': 0.6861321330070496},
   {'step': 12, 'loss': 0.6425641179084778},
   {'step': 13, 'loss': 0.6120173931121826},
   {'step': 14, 'loss': 0.572719156742096},
   {'step': 15, 'loss': 0.5421341061592102},
   {'step': 16, 'loss': 0.5096762776374817},
   {'step': 17, 'loss': 0.4803931415081024},
   {'step': 18, 'loss': 0.45265141129493713},
   {'step': 19, 'loss': 0.4256592094898224},
   {'step': 20, 'loss'

### Page 29

In [106]:
layer = nn.LayerNorm(3)
parameters = list(layer.named_parameters())
x = torch.tensor([1., 2, 3])
y = layer(x)
x = torch.tensor([100., 200, 300])
y = layer(x)

In [107]:
y

tensor([-1.2247,  0.0000,  1.2247], grad_fn=<NativeLayerNormBackward0>)